In [1]:
### Reading in the parameters from the par files

In [2]:
def norm_cumulative_diff(values, errors, wav, werr):
    """
    Cumulative shifts in the partition mu, normalized by the partition width.
    
    This is taken to mean the difference with respect to first value for the mu in the partition, normalized by (weighted) average sigma taken at the partition level.

    Parameters:
    - parameter1: values is the array of mus for which the difference needs to be calculated.
    - parameter2: errors associated with mus
    - parameter3: the (weighted) average sigma for the partition.
    - parameter4: error associated with the weighted average sigma at the partition-level

    Returns:
    Array contains elements given by mu_ij - mu_0j for every partition j
    """
    
    # array with the differences and errors is initialized
    norm_cum_diff     = []
    norm_cum_diff_unc = []      
        
    # If the first element is nan, set the reference to 0
    if not np.isnan(values[0]):
        ref_el  = values[0]
        ref_err = errors[0]
    else:
        ref_el  = 0
        ref_err = 0

    # Calculate the differences and errors
    for i, el in enumerate(values):
        cum_diff_value      = (el - ref_el)
        norm_cum_diff_value = cum_diff_value/wav[idx]

        cum_diff_unc_value      = np.sqrt(errors[i]**2 + ref_err**2)
        norm_cum_diff_unc_value = abs(norm_cum_diff_value) * np.sqrt ((cum_diff_unc_value/cum_diff_value)**2 + (werr[idx]/wav[idx])**2)
        
        norm_cum_diff.append(norm_cum_diff_value)
        norm_cum_diff_unc.append(norm_cum_diff_unc_value)
            
    return norm_cum_diff, norm_cum_diff_unc

In [3]:
def mod_diff(arr, err):
    """
    This is similar to the numpy.diff function but there is a small difference:
    If the previous point is NaN, the code will search for a non-NaN previous point to do the subtraction with.

    Parameters:
    - parameter1: arr is the array for which the difference between elements needs to be calculated.
    - parameter2: err is the array of errors associated with the elements

    Returns:
    - The array whose elements are the absolute difference between the elements of the original array and the associated errors.
    - The associated uncertainties.
    
    Note that the differences should be absolute values!
    """
    
    # if the first element is NaN, set previous element to 0, else set use it
    # set the error correspondingly
    if not np.isnan(arr[0]):
        prev_el  = arr[0]
        prev_err = err[0]
    else:
        prev_el  = 0
        prev_err = 0
        
    # array with the differences and errors is initialized
    diff     = []
    diff_unc = []
    
    # Calculate the differences and errors
    for i, curr_el in enumerate(arr):
        jk = i
        diff.append(abs(curr_el - prev_el))
        diff_unc.append(np.sqrt(err[i]**2 + prev_err**2))
        
        # Setting the next instance of the previous element
        if not np.isnan(curr_el):
            prev_el  = curr_el
            prev_err = err[i]
        else:
            while np.isnan(arr[jk]):
                jk      -= 1
                prev_el  = arr[jk]
                prev_err = err[jk]
                
    return diff, diff_unc

In [4]:
def weighted_average_and_uncertainty(values, errors):
    """
    Calculate the weighted average and its uncertainty when weights are the
    inverse of the uncertainty squared.

    Parameters:
    - values: NumPy array of values.
    - uncertainties: NumPy array of uncertainties corresponding to the values.

    Returns:
    - weighted_average
    """
    weighted_av     = []   
        
    # Slice the values and errors for the current segment
    segment_values = np.asarray(values, dtype=float)
    segment_errors = np.asarray(errors, dtype=float)
    
    if len(segment_values) != len(segment_errors):
        raise ValueError("The length of values and uncertainties must be the same.")

    # Calculate weights as the inverse of the uncertainty
    weights = 1 / (segment_errors)
    
    try:
        mask = ~np.isnan(segment_values)
        # Calculate weighted average
        weighted_av_value = np.average(segment_values[mask], weights=weights[mask])

        # Calculate uncertainty of the weighted average
        total_weight = np.sum(weights[mask])
        
    except:
        weighted_av_value = np.nan
    
    weighted_av.append(weighted_av_value)

    return weighted_av

In [5]:
def pull_datetimestamp(calfile):
    try:
        start_ind = calfile.index("-cal-") + len("-cal-")
        stop_ind = calfile.index("-tier")
    except:
        start_ind = calfile.index("-phy-") + len("-phy-")
        stop_ind = calfile.index("-par")
    dt = calfile[start_ind:stop_ind]
    return dt

In [6]:
def pull_chmap(dt):
    # dbetto change means that the if () statements need to be changed into the hasattr commands
    chmap = lmeta.hardware.configuration.channelmaps.on(dt)
    
    channel_dict_method = {}
    
    for channel_name, channel_data in chmap.items():     
        if channel_data.system == "geds":
            #print(channel_name)
            # Pull out the analysis flags based on the cal status
            ged = lmeta.channelmap(dt)[channel_name]
            ch=f'ch{ged["daq"]["rawid"]}'
            ged_ana = lmeta.datasets.statuses.on(dt, system="cal")[channel_name]

            channel_dict_method[ch] = (lmeta.channelmap(dt)[channel_name].type,
                                channel_name,
                                lmeta.channelmap(dt)[channel_name].production['mass_in_g'],
                                lmeta.channelmap(dt)[channel_name].location.string,
                                lmeta.channelmap(dt)[channel_name].location.position)
    return channel_dict_method

In [7]:
# Importing the packages
import json # to read/write the json files
import numpy as np # for numpy arrays
import pandas as pd # for pandas dataframes

from legendmeta import LegendMetadata # for querying the detector database
from datetime import datetime # for pulling channel maps relating to a specific time

import os
import glob

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import ticker

from functools import reduce

from matplotlib.patches import Rectangle

import re
from dbetto import TextDB, Props

import warnings
warnings.filterwarnings("ignore") # fed up of the load_dfs warnings

In [8]:
# Path to legend-metadata
# If pointing to a user specific path, remember to put it inside the paranthesis
lmeta = LegendMetadata()

could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f9291c73f60>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f9291c73ba0>)
could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f9291c911c0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f9291c90c20>)
could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f9291c9f650>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f9291c9f330>)
could not scan file /global/u2/j/jita/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <

In [9]:
runinfo = Props.read_from("/pscratch/sd/j/jita/legend_utils/legend-datasets/runinfo.yaml")
runlist = Props.read_from("/pscratch/sd/j/jita/legend_utils/legend-datasets/runlists.yaml")
cal_groupings = Props.read_from("/pscratch/sd/j/jita/legend_utils/legend-datasets/cal_groupings.yaml")

In [11]:
def expand_runs(in_dict):
    """
    This function expands out the runs if a range is specified in the dictionary
    e.g.
    {
        "p01": "r001..r005"
    }
    """
    for datatype, datalist in in_dict.items():
        for per, run_list in datalist.items():
            if isinstance(run_list, str) and ".." in run_list:
                start, end = run_list.split("..")
                in_dict[datatype][per] = [
                    f"r{x:03}" for x in range(int(start[1:]), int(end[1:]) + 1)
                ]
            if isinstance(run_list, list):
                for i, run in enumerate(run_list):
                    if isinstance(run, str) and ".." in run:
                        start, end = run.split("..")
                        run_list.pop(i)
                        expanded_runs = [
                            f"r{x:03}" for x in range(int(start[1:]), int(end[1:]) + 1)
                        ]
                        in_dict[datatype][per] += expanded_runs
                in_dict[datatype][per] = sorted(run_list)
    return in_dict

In [12]:
runlist = expand_runs(runlist["valid"])

In [14]:
file_tags = []
for pd in ['p16', 'p18', 'p19']:
    file_tags.extend([f'{pd}_{run}' for run in runlist['cal'][pd]])

In [15]:
basedir  = '/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/tmp/v3.1.0dev6/generated'
filt_tags = []
for file_tag in file_tags:
    try:
        parfile  = glob.glob(f'{basedir}/par/pht/cal/{file_tag[0:3]}/{file_tag[4:8]}/*-par_pht.yaml')[0]
        filt_tags.append(file_tag)
    except:
        continue

In [16]:
# Using the datetimestamp of March 2023
chmap = lmeta.hardware.configuration.channelmaps.on('20250911T235840Z')
channel_dict = {}
for channel_name, channel_data in chmap.items():
    if (channel_data.system == "geds"):
        channel_dict[channel_data["daq"]["rawid"]] = (channel_name,
                                                      channel_data.location.string,
                                                      channel_data.location.position,
                                                      channel_data.location.string*100 + channel_data.location.position)

In [17]:
import re
import numpy as np

def expand_runs(run_spec):
    if isinstance(run_spec, list):
        return run_spec
    m = re.fullmatch(r"r(\d+)\.\.r(\d+)", run_spec)
    if m:
        a, b = map(int, m.groups())
        w = len(m.group(1))
        return [f"r{i:0{w}d}" for i in range(a, b + 1)]
    return [run_spec]

def build_detector_partitions(det_name, calgroupings, min_prod=16):
    parts = {}

    # start from default
    for calgroup, prod_map in calgroupings.get("default", {}).items():
        keep = {}
        for p, run_spec in prod_map.items():
            if int(p[1:]) >= min_prod:
                keep[p] = expand_runs(run_spec)
        if keep:
            parts[calgroup] = keep

    # overwrite/add detector-specific partitions
    for calgroup, prod_map in calgroupings.get(det_name, {}).items():
        keep = {}
        for p, run_spec in prod_map.items():
            if int(p[1:]) >= min_prod:
                keep[p] = expand_runs(run_spec)
        if keep:
            parts[calgroup] = keep

    return parts

In [18]:
import pandas as pd
detlist = [channel_dict[key][0] for key in channel_dict.keys()]
# channel df
channel_df = pd.DataFrame.from_dict(channel_dict, orient='index',\
                                    columns=['channel_name',
                                             'string',
                                             'posn',
                                             'short_mageid'])

In [19]:
pardict = {}
for i, rawid in enumerate(channel_dict.keys()):
    det = channel_dict[rawid][0]
    parts = build_detector_partitions(det, cal_groupings)
    pardict[f'ch{rawid}']=parts

In [20]:
import glob
import yaml
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ---------------------------
# config
# ---------------------------
basedir = '/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/tmp/v3.1.0dev6/generated'
nan = np.nan

YamlLoader = yaml.UnsafeLoader

# partition_map should be your uploaded dict from turn1file0
partition_map = pardict   # e.g. {'ch1104004': {...}, ...}

# ---------------------------
# helpers
# ---------------------------
def make_nan_entry():
    return {'bl_std': nan, 'baselineEmax': nan}, nan, nan, nan, nan

def load_one_channel(data, det_name):
    try:
        results = data[det_name]['results']
        ecal = results['ecal']['monitoring_parameters']
        aoe = results['aoe']['1000-1300keV']
        ts = next(iter(aoe))
        vals = aoe[ts]

        bl = {
            'bl_std': ecal['bl_std']['mode'],
            'baselineEmax': ecal['baselineEmax']['mode'],
        }
        mu = vals['mean']
        muerr = vals['mean_err']
        sig = vals['sigma']
        sigerr = vals['sigma_err']
        return bl, mu, muerr, sig, sigerr
    except Exception:
        return make_nan_entry()

def mask_partition(filt_tags, values, members):
    return [v if tag in members else nan for tag, v in zip(filt_tags, values)]

def weighted_avg_and_mean_err(values, errors):
    v = np.asarray(values, dtype=float)
    e = np.asarray(errors, dtype=float)
    valid = np.isfinite(v) & np.isfinite(e) & (e > 0)
    if not np.any(valid):
        return nan, nan
    w = 1.0 / e[valid]**2
    avg = np.sum(w * v[valid]) / np.sum(w)
    err = np.nanmean(e[valid])
    return avg, err

# ---------------------------
# load all data
# ---------------------------
bl_pars = {}
df_mu = {}
df_muerr = {}
df_sig = {}
df_sigerr = {}

for file_tag in filt_tags:
    bl_pars[file_tag] = {}
    df_mu[file_tag] = {}
    df_muerr[file_tag] = {}
    df_sig[file_tag] = {}
    df_sigerr[file_tag] = {}

    pattern = f'{basedir}/par/pht/cal/{file_tag[:3]}/{file_tag[4:8]}/*-par_pht.yaml'
    parfile = next(glob.iglob(pattern), None)

    if parfile is None:
        for rawid in channel_dict:
            ch_key = f'ch{rawid}'
            bl, mu, muerr, sig, sigerr = make_nan_entry()
            bl_pars[file_tag][ch_key] = bl
            df_mu[file_tag][ch_key] = mu
            df_muerr[file_tag][ch_key] = muerr
            df_sig[file_tag][ch_key] = sig
            df_sigerr[file_tag][ch_key] = sigerr
            break
        continue

    with open(parfile, 'r') as fo:
        data = yaml.load(fo, Loader=YamlLoader)

    for rawid in channel_dict:
        ch_key = f'ch{rawid}'
        det_name = channel_dict[rawid][0]

        bl, mu, muerr, sig, sigerr = load_one_channel(data, det_name)

        bl_pars[file_tag][ch_key] = bl
        df_mu[file_tag][ch_key] = mu
        df_muerr[file_tag][ch_key] = muerr
        df_sig[file_tag][ch_key] = sig
        df_sigerr[file_tag][ch_key] = sigerr
        break

In [21]:
pdf_pages = PdfPages("Partition_checks_p16_p18_p19.pdf")

for rawid in channel_dict:
    ch_key = f'ch{rawid}'
    det_name = channel_dict[rawid][0]
    parts = partition_map.get(ch_key, {})

    print(f"Detector: {det_name} ✔")

    fig, axs = plt.subplots(nrows=3, ncols=2, sharex=True, figsize=(15, 8))
    fig.suptitle(
        f"RawID: {rawid}, Detector: {det_name}, "
        f"String: {channel_dict[rawid][1]}, Posn: {channel_dict[rawid][2]}"
    )

    mu_full = [df_mu[tag].get(ch_key, nan) for tag in filt_tags]
    muerr_full = [df_muerr[tag].get(ch_key, nan) for tag in filt_tags]
    sig_full = [df_sig[tag].get(ch_key, nan) for tag in filt_tags]
    sigerr_full = [df_sigerr[tag].get(ch_key, nan) for tag in filt_tags]
    blstd_full = [bl_pars[tag].get(ch_key, {}).get('bl_std', nan) for tag in filt_tags]
    bemax_full = [bl_pars[tag].get(ch_key, {}).get('baselineEmax', nan) for tag in filt_tags]

    wav = []
    werr = []

    for j, (calgroup, prod_map) in enumerate(parts.items()):
        members = set()
        for p, runs in prod_map.items():
            members.update(f"{p}-{r}" for r in runs)

        mu = mask_partition(filt_tags, mu_full, members)
        muerr = mask_partition(filt_tags, muerr_full, members)
        sig = mask_partition(filt_tags, sig_full, members)
        sigerr = mask_partition(filt_tags, sigerr_full, members)
        blstd = mask_partition(filt_tags, blstd_full, members)
        bemax = mask_partition(filt_tags, bemax_full, members)

        axs[0,0].errorbar(
            filt_tags, mu, yerr=muerr,
            marker='o', linestyle='--', label=calgroup
        )
        axs[0,1].errorbar(
            filt_tags, sig, yerr=sigerr,
            marker='s', linestyle='--', label=calgroup
        )
        axs[1,1].plot(
            filt_tags, blstd,
            marker='o', linestyle='--', label=calgroup
        )
        axs[2,1].plot(
            filt_tags, bemax,
            marker='o', linestyle='--', label=calgroup
        )

        avg_sig, avg_sigerr = weighted_avg_and_mean_err(sig, sigerr)
        wav.append(avg_sig)
        werr.append(avg_sigerr)

    axs[0,0].set_ylabel(r'$\mu$ [ADC]')
    axs[0,1].set_ylabel(r'$\sigma$ [ADC]')
    axs[1,1].set_ylabel('bl_std')
    axs[2,1].set_ylabel('baselineEmax')
    axs[2,0].set_xlabel('Run')
    axs[2,1].set_xlabel('Run')

    for ax in axs.flat:
        ax.grid(linestyle='dashed', linewidth=0.5)

    axs[0,0].legend(loc='upper left', fontsize=8)
    axs[0,1].legend(loc='upper right', fontsize=8)
    axs[1,1].legend(loc='upper right', fontsize=8)
    axs[2,1].legend(loc='upper right', fontsize=8)

    axs[2,0].axis('off')  # until you rework slow-shift metric per partition

    for ax in axs.flat:
        ax.tick_params(axis='x', labelrotation=90)

    plt.tight_layout()
    pdf_pages.savefig(fig)
    plt.close(fig)
    break

pdf_pages.close()

Detector: V14654A ✔
